In [ ]:
#SOTA Post-2024 Spatiotemporal Semivariogram (to be compared)

In [1]:
#0) Install & upload

!pip -q install numpy pandas openpyxl scipy libpysal spreg

from google.colab import files
files.upload()  # upload: data.xlsx, matrices.xlsx

Saving matrices_JarakCentroid_10th.xlsx to matrices_JarakCentroid_10th.xlsx
Saving data_JarakCentroid_10th_Pendidikan.xlsx to data_JarakCentroid_10th_Pendidikan.xlsx


{'matrices_JarakCentroid_10th.xlsx': b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00zo,\xc9n\x01\x00\x00\xe4\x05\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

In [2]:
#1) Load data & matrices

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix, kron
from scipy.optimize import curve_fit

import libpysal
from spreg import ML_Lag, GM_Lag

DATA_FILE   = "data_JarakCentroid_10th_Pendidikan.xlsx"
MATRIX_FILE = "matrices_JarakCentroid_10th.xlsx"

df = pd.read_excel(DATA_FILE)
xls = pd.ExcelFile(MATRIX_FILE)
print("Sheets:", xls.sheet_names)
print("df:", df.shape, df.columns.tolist())

def read_mat(sheet):
    return pd.read_excel(MATRIX_FILE, sheet_name=sheet, header=None).to_numpy(dtype=float)

# shared matrices
W_T = read_mat("W_T")  # 10x10
I_T = read_mat("I_T")  # 10x10
I_S = read_mat("I_S")  # 35x35

# distance matrix (required)
D   = read_mat("D")    # 35x35

assert df.shape[0] == 350
assert W_T.shape == (10,10)
assert I_T.shape == (10,10)
assert I_S.shape == (35,35)
assert D.shape   == (35,35)

n, T = 35, 10

Sheets: ['W_T', 'I_T', 'I_S', 'D']
df: (350, 4) ['Waktu', 'Lokasi', 'Y', 'Y_(t-1)']


In [3]:
#2) Build Y matrix (Tn) according to the data stacking order (blocks per year)

years_order = list(pd.unique(df["Waktu"]))
print("years_order:", years_order)
assert len(years_order) == T

y_vec = df["Y"].astype(float).to_numpy()
Y = y_vec.reshape(T, n)
print("Y shape:", Y.shape)

years_order: [np.int64(2023), np.int64(2022), np.int64(2021), np.int64(2020), np.int64(2019), np.int64(2018), np.int64(2017), np.int64(2016), np.int64(2015), np.int64(2014)]
Y shape: (10, 35)


In [4]:
#3) Empirical variogram (within-year) + fit exponential variogram

def empirical_variogram_within_year(Y, D, n_bins=12):
    """
    Pool semivariance across years using within-year pairs (u=0).
    Returns bin centers (h) and empirical gamma(h) per bin.
    """
    T, n = Y.shape
    # standardize per year
    Z = (Y - Y.mean(axis=1, keepdims=True)) / Y.std(axis=1, keepdims=True)

    hs, gs = [], []
    for t in range(T):
        z = Z[t]
        for i in range(n):
            for j in range(i+1, n):
                hs.append(D[i, j])
                gs.append(0.5 * (z[i] - z[j])**2)

    hs = np.array(hs); gs = np.array(gs)

    bins = np.linspace(hs.min(), hs.max(), n_bins+1)
    bin_id = np.digitize(hs, bins) - 1

    h_centers, gamma_emp = [], []
    for b in range(n_bins):
        m = bin_id == b
        if m.sum() < 10:
            continue
        h_centers.append(hs[m].mean())
        gamma_emp.append(gs[m].mean())

    return np.array(h_centers), np.array(gamma_emp)

# exponential model: gamma(h)= nugget + sill*(1-exp(-h/a))
def exp_variogram(h, nugget, sill, a):
    return nugget + sill * (1.0 - np.exp(-h / a))

h_c, gamma_c = empirical_variogram_within_year(Y, D, n_bins=12)
print("Empirical variogram points:", len(h_c))

p0 = [max(1e-6, gamma_c.min()*0.5), max(1e-6, gamma_c.max()), max(1e-6, np.median(h_c))]
bounds = ([0, 0, 1e-6], [np.inf, np.inf, np.inf])

params, _ = curve_fit(exp_variogram, h_c, gamma_c, p0=p0, bounds=bounds, maxfev=50000)
nugget, sill, a = params
range_eff = 3*a  # effective range heuristic
print("Fitted exp variogram:", {"nugget": nugget, "sill": sill, "a": a, "effective_range~": range_eff})

Empirical variogram points: 12
Fitted exp variogram: {'nugget': np.float64(0.12164747484020447), 'sill': np.float64(1.0512548083126725), 'a': np.float64(42.055341768196456), 'effective_range~': np.float64(126.16602530458937)}


In [5]:
#4) Build WSV,bin: binary by variogram range + row-normalization

def build_WS_binary_from_range(D, range_eff, row_normalize=True):
    n = D.shape[0]
    WS = (D <= range_eff).astype(float)
    np.fill_diagonal(WS, 0.0)

    if row_normalize:
        rs = WS.sum(axis=1, keepdims=True)
        WS = np.divide(WS, rs, out=np.zeros_like(WS), where=rs != 0)
    return WS

W_S_vario_bin = build_WS_binary_from_range(D, range_eff, row_normalize=True)
print("W_S_vario_bin:", W_S_vario_bin.shape, "row sum min/max:", W_S_vario_bin.sum(1).min(), W_S_vario_bin.sum(1).max())

W_S_vario_bin: (35, 35) row sum min/max: 0.9999999999999993 1.0000000000000004


In [6]:
#5) Build WSTV,bin via Kronecker Eq. (10)

def build_WST(W_S, W_T, I_S, I_T):
    W_ST_sp = kron(csr_matrix(I_T), csr_matrix(W_S), format="csr") + \
              kron(csr_matrix(W_T), csr_matrix(I_S), format="csr")
    W_ST = W_ST_sp.toarray()
    np.fill_diagonal(W_ST, 0.0)
    return W_ST

W_ST_vario_bin = build_WST(W_S_vario_bin, W_T, I_S, I_T)
print("W_ST_vario_bin:", W_ST_vario_bin.shape)

W_ST_vario_bin: (350, 350)


In [7]:
##OUTPUT 1: Moran-2Di inference (within-year permutation)
#6) Moran-2Di + Monte Carlo permutation → I_obs, I_mean, p, CI95

def moran_2di(z_vec, W_ST):
    z = np.asarray(z_vec, dtype=float).reshape(-1, 1)
    N = z.shape[0]
    S0 = float(np.sum(W_ST))
    if S0 == 0:
        raise ValueError("S0 = sum(W_ST) is zero.")
    num = float(z.T @ W_ST @ z)
    den = float(z.T @ z)
    return (N / S0) * (num / den)

def mc_within_year_permutation_moran2di(z, years, W_ST, B=1000, seed=2026, two_sided=True):
    rng = np.random.default_rng(seed)
    z = np.asarray(z, dtype=float).ravel()
    years = np.asarray(years).ravel()

    I_obs = moran_2di(z, W_ST)

    uniq = np.unique(years)
    idxs = {yr: np.where(years == yr)[0] for yr in uniq}

    I_perm = np.empty(B, dtype=float)
    for b in range(B):
        z_perm = z.copy()
        for yr, idx in idxs.items():
            z_perm[idx] = rng.permutation(z_perm[idx])
        I_perm[b] = moran_2di(z_perm, W_ST)

    I_mean = float(I_perm.mean())
    ci_low, ci_high = float(np.quantile(I_perm, 0.025)), float(np.quantile(I_perm, 0.975))

    if two_sided:
        p_value = (np.sum(np.abs(I_perm) >= np.abs(I_obs)) + 1) / (B + 1)
    else:
        p_value = (np.sum(I_perm >= I_obs) + 1) / (B + 1)

    return {"I_obs": float(I_obs), "I_mean": I_mean, "p_value": float(p_value),
            "CI95_low": ci_low, "CI95_high": ci_high, "I_perm": I_perm}

y_all = df["Y"].astype(float).to_numpy()
z = (y_all - y_all.mean()) / y_all.std(ddof=0)
years_vec = df["Waktu"].to_numpy()

out_moran_bin = mc_within_year_permutation_moran2di(z, years_vec, W_ST_vario_bin, B=1000, seed=2026, two_sided=True)

print("=== Moran-2Di (Variogram Binary baseline) ===")
print("I_obs   :", out_moran_bin["I_obs"])
print("I_mean  :", out_moran_bin["I_mean"])
print("p-value :", out_moran_bin["p_value"])
print("CI95    :", (out_moran_bin["CI95_low"], out_moran_bin["CI95_high"]))

=== Moran-2Di (Variogram Binary baseline) ===
I_obs   : 0.43229245351089673
I_mean  : 0.021882033485870662
p-value : 0.000999000999000999
CI95    : (-0.028742892901651117, 0.07233766865016329)


/tmp/ipykernel_419/2031927385.py:10: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  num = float(z.T @ W_ST @ z)
/tmp/ipykernel_419/2031927385.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  den = float(z.T @ z)


In [8]:
#OUTPUT 2: Eq. (13) evaluation (spatial block bootstrap by location)
#7) Bootstrap by location → R2_mean, RMSE_mean (+CI95)

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    return np.sqrt(np.mean((y_true - y_pred)**2))

def r2(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ss_res/ss_tot if ss_tot > 0 else np.nan

def fit_eq13_model(y, X, Wmat, shrink=0.99):
    Wmat = np.array(Wmat, dtype=float, copy=True)
    np.fill_diagonal(Wmat, 0.0)

    Ws = sp.csr_matrix(Wmat)
    w = libpysal.weights.WSP(Ws); w.transform = "R"
    w = libpysal.weights.WSP(w.sparse * shrink); w.transform = "R"

    try:
        m = ML_Lag(y=y, x=X, w=w, name_y="Y", name_x=["Y_t-1","const"])
    except Exception:
        m = GM_Lag(y=y, x=X, w=w, name_y="Y", name_x=["Y_t-1","const"])
    return m

def spatial_block_bootstrap_by_location(Y, W_ST, B=1000, seed=2026, shrink=0.99):
    rng = np.random.default_rng(seed)
    T, n = Y.shape
    N = T*n

    # build observed y and lag from Y
    Y_lag = np.zeros_like(Y)
    Y_lag[1:, :] = Y[:-1, :]
    y_obs = Y.reshape(N, 1)
    xlag_obs = Y_lag.reshape(N, 1)
    X_obs = np.hstack([xlag_obs, np.ones((N, 1))])

    # location blocks: keep all years for each location
    loc_blocks = [np.array([i + t*n for t in range(T)]) for i in range(n)]

    out = []
    failed = 0

    for b in range(1, B+1):
        chosen = rng.choice(np.arange(n), size=n, replace=True)
        idx = np.concatenate([loc_blocks[i] for i in chosen])  # length N

        y_b = y_obs[idx, :]
        X_b = X_obs[idx, :]
        W_b = W_ST[np.ix_(idx, idx)].copy()
        np.fill_diagonal(W_b, 0.0)

        try:
            model = fit_eq13_model(y_b, X_b, W_b, shrink=shrink)
            yhat = model.predy
            out.append({
                "iter": b,
                "rho": float(model.rho),
                "RMSE": float(rmse(y_b, yhat)),
                "R2": float(r2(y_b, yhat))
            })
        except Exception:
            failed += 1

    return pd.DataFrame(out), failed

boot_bin, failed = spatial_block_bootstrap_by_location(Y, W_ST_vario_bin, B=1000, seed=2026, shrink=0.99)
print("Bootstrap success:", len(boot_bin), "failed:", failed)

R2_mean = float(boot_bin["R2"].mean())
RMSE_mean = float(boot_bin["RMSE"].mean())
print("R2_mean   :", R2_mean)
print("RMSE_mean :", RMSE_mean)

print("R2_CI95   :", (float(boot_bin["R2"].quantile(0.025)), float(boot_bin["R2"].quantile(0.975))))
print("RMSE_CI95 :", (float(boot_bin["RMSE"].quantile(0.025)), float(boot_bin["RMSE"].quantile(0.975))))

/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: User

GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag


/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of 

GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag


/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: User

GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag
GM_Lag


... (216 outputs left out)

/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of 

GM_Lag
GM_Lag
GM_Lag


/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "rho": float(model.rho),
/usr/local/lib/python3.13/dist-packages/spreg/user_output.py:611: UserWarning: w must be API-compatible pysal weights object
  warn("w must be API-compatible pysal weights object")
/tmp/ipykernel_419/680406117.py:62: DeprecationWarning: Conversion of 

GM_Lag
Bootstrap success: 1000 failed: 0
R2_mean   : 0.04069136675093988
RMSE_mean : 2.5234807107778408
R2_CI95   : (0.0010964607616530538, 0.1617727970123286)
RMSE_CI95 : (2.0784583592446713, 2.975382201596109)


In [9]:
#8) (Optional) Export summary for supplementary
summary_bin = pd.DataFrame([{
    "baseline": "Variogram Binary (range-threshold, row-normalized)",
    "effective_range": float(range_eff),
    "I_obs": out_moran_bin["I_obs"],
    "I_mean": out_moran_bin["I_mean"],
    "p_value": out_moran_bin["p_value"],
    "CI95_low": out_moran_bin["CI95_low"],
    "CI95_high": out_moran_bin["CI95_high"],
    "R2_mean": R2_mean,
    "RMSE_mean": RMSE_mean,
}])

summary_bin.to_csv("variogram_binary_summary_Centroid_1000rep_10th_Pendidikan.csv", index=False)
boot_bin.to_csv("variogram_binary_bootstrap_details_Centroid_1000rep_10th_Pendidikan.csv", index=False)

from google.colab import files
files.download("variogram_binary_summary_Centroid_1000rep_10th_Pendidikan.csv")
files.download("variogram_binary_bootstrap_details_Centroid_1000rep_10th_Pendidikan.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>